# SemanticPrism: Master Workflow Pipeline

This notebook serves as the definitive orchestrator for the entire SemanticPrism extraction architecture.
It heavily documents every phase, handles deep generative synthesis via Pydantic AI, and natively 
exports intermediate logical matrices into structured output directories for full diagnostic transparency.

By running this notebook cell by cell, you can visually trace how logic is extracted, normalized, mathematically embedded, taxonomically mapped, topologically structured, and finally synthesized into discrete code blocks.


## Imports & Helper Functions
We begin by setting up the global namespace, importing the underlying native components of the SemanticPrism pipeline, and defining a rigorous `_save_state` hook to physically persist Pydantic AI validation matrices to disk at each step.


In [ ]:
import os
import yaml
import json
import glob
import time
import asyncio
from typing import Any

# Import Core Engines
from src.core.logger import get_logger
from src.extraction.extractor import ExtractionPipeline
from src.extraction.normalize_text import execute_normalization_phase
from src.embedding.embedding import EmbeddingPipeline
from src.nlp.hypernyms import HypernymPipeline
from src.nlp.nlp_mapping import NamingResolutionPipeline
from src.topology.graph_builder import TopologyEngine
from src.synthesis.synthesizer import SynthesisEngine
from src.helpers.visualizer import SemanticVisualizer

logger = get_logger("MasterNotebook")

def _save_state(data: Any, filepath: str):
    """
    Safely serializes Pydantic schemas, Python Sets, and Base Dictionaries natively to disk.
    """
    os.makedirs(os.path.dirname(filepath), exist_ok=True)
    with open(filepath, 'w', encoding='utf-8') as f:
        default_encoder = lambda x: list(x) if isinstance(x, set) else str(x)
        if isinstance(data, list) and len(data) > 0 and hasattr(data[0], 'model_dump'):
            json.dump([item.model_dump(mode='json') for item in data], f, indent=4, default=default_encoder)
        elif hasattr(data, 'model_dump'):
            json.dump(data.model_dump(mode='json'), f, indent=4, default=default_encoder)
        else:
            json.dump(data, f, indent=4, default=default_encoder)
            
print("✅ Core imports loaded successfully.")


## Phase 0: Ingestion & Engine Instantiation
Before we begin the logical processes, we load the unstructured `.txt` and `.md` corpus files from the `inputs/testdocs/` directory.

We then initialize our underlying classes (which instantiate LLMs, load offline SentenceTransformer embeddings, and set up the NetworkX matrix processors).


In [ ]:
# Load Full Text of Reports
import polars as pl
import random

HBN = pl.read_parquet("../../orig_wAnon_HBN.parquet")
#HBN.head()
HBN_Anon = HBN.get_column("anonymized")
HBN_Reports = HBN_Anon.to_list()

print(len(HBN_Reports))

selected_items = random.sample(HBN_Reports, 6)

for i, item in enumerate(selected_items, start=1):
    file_name = f"./inputs/HBN/output_{i}.txt"
    with open(file_name, "w") as file:
        file.write(item)
    #print(f"Saved '{item}' to {file_name}")


In [ ]:
target_dir = "inputs/HBN"
files = glob.glob(os.path.join(target_dir, "*.txt")) + glob.glob(os.path.join(target_dir, "*.md"))

raw_texts = []
for path in files:
    with open(path, "r", encoding="utf-8") as f:
        raw_texts.append(f.read())
        
print(f"Ingested {len(raw_texts)} logic documents natively.")

# Instantiate all underlying mathematical and Generative engines
extractor = ExtractionPipeline("config.yaml")
embedder = EmbeddingPipeline("config.yaml")
hypernyms = HypernymPipeline("config.yaml")
mapper = NamingResolutionPipeline()
topology = TopologyEngine()
synthesizer = SynthesisEngine("config.yaml")
visualizer = SemanticVisualizer()

print("✅ Mathematical and Generative Engines initialized.")


## Phase 1: Discovery & Thematic Extraction
In this phase, we interact with the instantiated LLM via Pydantic AI to:
1. Discover overarching sub-themes iteratively across every document.
2. Mathematically weigh them by frequency to synthesize a unifying **Master Domain**.
3. Extract concrete Subject-Verb-Object (SVO) logic triples gated against that Master Domain.
4. Execute Lexical Normalization to strip away stop words and noise.

*Note: We use `await` natively in notebook cells because Jupyter supports async.*


In [ ]:
# Process 1.A: Parallel Sub-Thematic Discovery
all_themes = []
for idx, text in enumerate(raw_texts):
    print(f"Discovering thematic nodes inside document {idx + 1}/{len(raw_texts)}")
    themes = await extractor.discover_themes(text)
    all_themes.extend(themes)
    
_save_state(all_themes, "outputs/01_extraction/original_themes.json")

# Process 1.B: Master Domain Synthesis
weighted_string = extractor.weight_themes(all_themes)
master_context = await extractor.consolidate_themes(weighted_string)
master_domain = master_context.master_domain if master_context else "General Complex Logic"
_save_state(master_context, "outputs/01_extraction/distilled_themes.json")
print(f"\n🎯 Master Domain Distilled: {master_domain}")

In [ ]:
# Process 1.C: Logical Triple Extraction (SVO)
raw_triples = []
for idx, text in enumerate(raw_texts):
    print(f"Extracting factual logic triples from document {idx + 1}/{len(raw_texts)}")
    trips = await extractor.extract_triples(text, master_context)
    raw_triples.extend(trips)
    
_save_state(raw_triples, "outputs/01_extraction/original_triplets.json")

# Generate visualization of raw extractions
visualizer.visualize_triples(raw_triples, "outputs/01_extraction/01_raw_triples_graph.html", "Phase 1: Raw Extractions")

In [ ]:
# Process 1.D: Text Normalizationtly overrides Pydantic AI's default behavior, forcing the agent to use standard structured JSON output instead of attempting to use tool calls. Give it another run and it should correctly interface with the model n
print("\nExecuting NLP Lexical Normalization arrays...")
normalized_triples, _, _, _ = await execute_normalization_phase(
    extractor, raw_triples, master_domain, _save_state
)

print(f"✅ Extracted {len(raw_triples)} raw logic facts and normalized them natively.")

### Phase 1 Visualization
You can view the raw HTML projection outputted from Phase 1 using an iframe.


In [ ]:
from IPython.display import IFrame
IFrame(src='outputs/01_extraction/01_raw_triples_graph.html', width='100%', height=500)


## Phase 2: Offline Embedding Compression
To reduce massive string comparisons into physical math, we map the normalized triples into high-dimensional Euclidean space. 
We then utilize an Eigen-gap analysis (PCA dimensionality reduction) and cluster the spatial nodes utilizing Agglomerative Cosine metrics.


In [ ]:
# Map text to embeddings -> PCA compress -> Cosine cluster
proposed_clusters = embedder.process_triples(normalized_triples)
_save_state(proposed_clusters, "outputs/02_embedding/clusters_identified.json")

print(f"✅ Mathematical embedding compression generated {len(proposed_clusters)} distinct physical clusters.")


## Phase 3: Hybrid Taxonomic Lifting
We cannot blindly trust spatial mathematics to correlate semantic truth. 
1. The **Contextual Validation** LLM pass gates each physical cluster to ensure they actually align logically.
2. The **Taxonomic Lift** mechanism abstracts the verified nodes into a higher-order concept (Hypernym).


In [ ]:
# Process 3.A: Contextual Validation
verified_clusters = await hypernyms.validate_context_vectors(proposed_clusters, master_domain)
_save_state(verified_clusters, "outputs/03_hypernym_lifting/verified_clusters.json")

# Process 3.B: Taxonomic Lift (Hypernym creation)
hypernym_mapping = await hypernyms.taxonomic_lift(verified_clusters, master_domain)
_save_state(hypernym_mapping, "outputs/03_hypernym_lifting/hypernym_mapping.json")

print(f"✅ Lifted {len(hypernym_mapping)} overarching taxonomy hypernyms securely.")


## Phase 4: Taxonomic Naming Resolution
We now traverse the raw array of semantic logic facts and systematically replace the noisy strings with the crisp, abstracted hypernyms we just established.
This dramatically collapses graph sparsity!


In [ ]:
# Resolving taxonomy math
mapped_triples = mapper.resolve_names(normalized_triples, hypernym_mapping)
_save_state(mapped_triples, "outputs/04_mapping/mapped_triplets.json")

# Visualize the newly simplified taxonomy graph
visualizer.visualize_triples(mapped_triples, "outputs/04_mapping/02_resolved_triples_graph.html", "Phase 4: Abstracted Taxonomy")

print("✅ Mapped abstracts onto triples securely.")


### Phase 4 Visualization


In [ ]:
IFrame(src='outputs/04_mapping/02_resolved_triples_graph.html', width='100%', height=500)


## Phase 5: Topological Network Analysis
Having established the simplified connections, we transition to topological operations:
1. Building a `NetworkX` DiGraph to prune duplicates mathematically.
2. Utilizing the `Leiden` modularity algorithm to find localized sub-communities.
3. Calculating **N-ary Hypergraphs** to establish Bipartite Incidence ($H$) and Laplacian ($L$) physical matrices.


In [ ]:
# 5.A: Construct DiGraph
graph = topology.build_graph(mapped_triples)

# Load Topology configurations safely mathematically
overlap_threshold = 0.80
leiden_resolution = 1.0
min_community_size = 4
if os.path.exists("config.yaml"):
    with open("config.yaml", "r") as f:
        cfg = yaml.safe_load(f)
        overlap_threshold = cfg.get("topology", {}).get("inheritance_overlap_threshold", 0.80)
        leiden_resolution = cfg.get("topology", {}).get("leiden_resolution", 1.0)
        min_community_size = cfg.get("topology", {}).get("min_community_size", 4)

# 5.B: Community Detection
partition = topology.detect_communities(graph, resolution=leiden_resolution)
hierarchy = topology.extract_hierarchy(graph, partition, min_size=min_community_size)

_save_state(partition, "outputs/05_topology/modularity_partition.json")
_save_state(hierarchy, "outputs/05_topology/extracted_hierarchy.json")

visualizer.visualize_topology(graph, partition, "outputs/05_topology/03_topology_communities_graph.html", "Phase 5: Modularity Topology")

# 5.C: N-ary Hypergraph Calculation
hyper_data = topology.build_hypergraph_topology(mapped_triples, overlap_threshold=overlap_threshold)
topology.visualize_hypergraph(hyper_data["B"], "outputs/05_topology")

hyper_export = {
    "entities_count": hyper_data["entities"],
    "themes_count": hyper_data["themes"],
    "H_matrix": hyper_data["H"].tolist(),
    "L_matrix": hyper_data["L"].tolist(),
    "theme_inheritance_map": hyper_data.get("theme_inheritance_map", {})
}
_save_state(hyper_export, "outputs/05_topology/hypergraph_matrices.json")

print(f"✅ Modularity executed and H/L matrices computed natively.")


### Phase 5 Visualization


In [ ]:
IFrame(src='outputs/05_topology/03_topology_communities_graph.html', width='100%', height=600)


## Phase 6: Generative Synthesis
We feed the isolated topological communities directly to the synthesis agent, demanding Pydantic Schema validations matching exactly to structured classes. We merge the final artifacts into our ultimate python definitions natively!


In [ ]:
# Parses the discrete Leiden communities into structural Pydantic models cleanly
theme_inheritance_map = hyper_data.get("theme_inheritance_map", {})
resolved_schemas = await synthesizer.generate_schemas(hierarchy, master_domain, theme_inheritance_map)

# Formats the final structural Python class code blocks and writes them logically
file_path = synthesizer.build_global_context(resolved_schemas)

print(f"🎉 Pipeline Complete! Final logical schema outputs synthesized securely to: {file_path}")
